# Lezione 13 · Le altre scuole

La restauratrice e il suo bozzetto, il gioco delle cartoline, le parole restaurate, la busta e il foglietto, i vecchi al bar del porto, la forchetta, e le linee tra le barche.

Ogni numero calcolato qui viene confrontato con quello scritto nel libro: un segno ✓ vuol dire che coincidono.

In [1]:
import math
import numpy as np

def controlla(nome, valore, atteso, tolleranza=0.01):
    """Confronta un valore calcolato con quello scritto nel libro."""
    ok = abs(valore - atteso) <= tolleranza
    segno = "✓" if ok else "✗ DIVERSO DAL LIBRO"
    print(f"{segno}  {nome}: calcolato {valore:.4g}, nel libro {atteso:g}")
    assert ok, nome

def conta(nome, valore, atteso):
    """Come controlla, per i numeri interi: devono coincidere esattamente."""
    ok = valore == atteso
    segno = "✓" if ok else "✗ DIVERSO DAL LIBRO"
    intero = lambda x: f"{x:,.0f}".replace(",", ".")
    print(f"{segno}  {nome}: calcolato {intero(valore)}, nel libro {intero(atteso)}")
    assert ok, nome

## La restauratrice in numeri

*Nel racconto: sporcare e imparare a togliere il rumore; venti, cinquanta passate.*

In allenamento si prende un'immagine vera x₀ e la si sporca: xₜ = √ᾱₜ · x₀ + √(1 − ᾱₜ) · ε. Quando ᾱₜ scende da quasi 1 a quasi 0, l'immagine lascia il posto al rumore.

In [2]:
for alfa in [0.99, 0.5, 0.01]:
    immagine, rumore = math.sqrt(alfa), math.sqrt(1 - alfa)
    print(f"ᾱ = {alfa:<4}: xₜ = {immagine:.2f} · x₀ + {rumore:.2f} · ε")
    assert math.isclose(immagine**2 + rumore**2, 1)          # i due pesi, al quadrato, fanno sempre 1

passate = 30
print(f"un'immagine in {passate} passate chiede {passate} letture della rete che pulisce")

ᾱ = 0.99: xₜ = 0.99 · x₀ + 0.10 · ε
ᾱ = 0.5 : xₜ = 0.71 · x₀ + 0.71 · ε
ᾱ = 0.01: xₜ = 0.10 · x₀ + 0.99 · ε
un'immagine in 30 passate chiede 30 letture della rete che pulisce


Per risparmiare si pulisce un bozzetto invece dell'immagine.

In [3]:
immagine = 512 * 512 * 3             # pixel per pixel, tre colori
bozzetto = 64 * 64 * 4               # quattro canali
conta("numeri dell'immagine", immagine, 786_432)
conta("numeri del bozzetto", bozzetto, 16_384)
controlla("quante volte meno", immagine / bozzetto, 48, 0)

✓  numeri dell'immagine: calcolato 786.432, nel libro 786.432
✓  numeri del bozzetto: calcolato 16.384, nel libro 16.384
✓  quante volte meno: calcolato 48, nel libro 48


## Il gioco delle cartoline

*Nel racconto: le dieci cartoline e le dieci didascalie; quattrocento milioni di immagini nel 2021; le tessere di sedici pixel.*

In un mucchio di N coppie si calcola la somiglianza di ogni immagine con ogni frase, una tabella N × N. Le coppie giuste stanno sulla diagonale.

In [4]:
N = 10
conta("le dieci cartoline: caselle", N * N, 100)
conta("le dieci cartoline: coppie giuste, sulla diagonale", N, 10)

N = 32_768
print(f"un mucchio di {N:,} coppie: {N * N:,} caselle".replace(",", "."))
assert N * N > 1e9                               # «oltre un miliardo di caselle»
conta("coppie sbagliate che ogni coppia giusta deve battere", N - 1, 32_767)

✓  le dieci cartoline: caselle: calcolato 100, nel libro 100
✓  le dieci cartoline: coppie giuste, sulla diagonale: calcolato 10, nel libro 10
un mucchio di 32.768 coppie: 1.073.741.824 caselle
✓  coppie sbagliate che ogni coppia giusta deve battere: calcolato 32.767, nel libro 32.767


Le tessere: un'immagine di 224 × 224 pixel, tagliata in tessere di 16 × 16. E la voce: un vettore ogni 20 millesimi di secondo.

In [5]:
per_lato = 224 // 16
conta("tessere per lato", per_lato, 14)
conta("tessere, cioè token", per_lato * per_lato, 196)
conta("token per trenta secondi di voce", 30 * 1000 // 20, 1500)

✓  tessere per lato: calcolato 14, nel libro 14
✓  tessere, cioè token: calcolato 196, nel libro 196
✓  token per trenta secondi di voce: calcolato 1.500, nel libro 1.500


## Le parole restaurate

*Nel racconto: le otto caselle coperte; il viaggio fatto una volta per molte caselle.*

Un blocco di 256 token scritto in fila chiede 256 passaggi del modello; scoperto in 32 passate, ne chiede 32.

In [6]:
controlla("quante volte meno viaggi dei pesi", 256 / 32, 8, 0)

✓  quante volte meno viaggi dei pesi: calcolato 8, nel libro 8


Il guaio della funivia: due caselle vicine, ciascuna con due parole possibili, metà e metà, una da frase di montagna e una da frase di mare. Scoperte nella stessa passata, ognuna sceglie per conto suo.

In [7]:
from itertools import product
combinazioni = list(product(["montagna", "mare"], repeat=2))
stonate = [c for c in combinazioni if c[0] != c[1]]
for c in combinazioni:
    print(c, "" if c[0] == c[1] else "← non stanno insieme")
conta("combinazioni ugualmente probabili", len(combinazioni), 4)
conta("combinazioni che non stanno insieme", len(stonate), 2)

('montagna', 'montagna') 
('montagna', 'mare') ← non stanno insieme
('mare', 'montagna') ← non stanno insieme
('mare', 'mare') 
✓  combinazioni ugualmente probabili: calcolato 4, nel libro 4
✓  combinazioni che non stanno insieme: calcolato 2, nel libro 2


## La busta e il foglietto

*Nel racconto: oltre 40 GB per una conversazione lunga; la busta in uno strato su quattro; il foglietto che costa sempre uguale.*

Il modello da 70 miliardi della nave di linea lascia nella busta 327.680 byte per ogni token.

In [8]:
byte_per_token = 2 * 80 * 8 * 128 * 2
conta("byte per token", byte_per_token, 327_680)
busta = byte_per_token * 128_000 / 1e9
controlla("busta a 128.000 token (GB)", busta, 42, 0.1)
controlla("busta in uno strato su quattro (GB)", busta / 4, 10.5, 0.05)

foglietto = 80 * 16_384 * 128 * 2    # 80 strati, 16.384 canali, 128 numeri per canale, a 16 bit
controlla("foglietto del modello a stato (milioni di byte)", foglietto / 1e6, 335, 1)
controlla("foglietto (GB)", foglietto / 1e9, 0.3, 0.05)
print("e resta così, qualunque sia la lunghezza della pagina")

✓  byte per token: calcolato 327.680, nel libro 327.680
✓  busta a 128.000 token (GB): calcolato 41.94, nel libro 42
✓  busta in uno strato su quattro (GB): calcolato 10.49, nel libro 10.5
✓  foglietto del modello a stato (milioni di byte): calcolato 335.5, nel libro 335
✓  foglietto (GB): calcolato 0.3355, nel libro 0.3
e resta così, qualunque sia la lunghezza della pagina


L'aggiornamento lineare: hₜ = A · hₜ₋₁ + B · xₜ. Due passi di fila danno hₜ = A² · hₜ₋₂ + A · B · xₜ₋₁ + B · xₜ: i conti si possono riordinare. Qui A, B e i numeri in arrivo sono tirati a caso.

In [9]:
rng = np.random.default_rng(0)
A, B = rng.normal(size=(4, 4)), rng.normal(size=(4, 3))
h, x1, x2 = rng.normal(size=4), rng.normal(size=3), rng.normal(size=3)

in_fila = A @ (A @ h + B @ x1) + B @ x2          # un passo, poi l'altro
riordinato = A @ A @ h + A @ B @ x1 + B @ x2     # tutto insieme
assert np.allclose(in_fila, riordinato)
print("✓  due aggiornamenti di fila si fondono in uno solo")

✓  due aggiornamenti di fila si fondono in uno solo


## L'albero e i vecchi al bar

*Nel racconto: la Stella, le tre domande e la foglia «una su cinque»; i tre vecchi, 30 + 15 + 5.*

La Stella: nove metri e mezzo, motore nuovo, vento a venticinque nodi.

In [10]:
stella = {"vento": 25, "lunghezza": 9.5, "anni del motore": 0}
risposte = [stella["vento"] > 20, stella["lunghezza"] < 9, stella["anni del motore"] > 15]
print("vento sopra i venti nodi? barca sotto i nove metri? motore con più di quindici anni?")
print(["sì" if r else "no" for r in risposte])
assert risposte == [True, False, False]          # sì, no, no: come nel racconto
controlla("la foglia: in ritardo una su cinque", 1 / 5, 0.20, 0)

vento sopra i venti nodi? barca sotto i nove metri? motore con più di quindici anni?
['sì', 'no', 'no']
✓  la foglia: in ritardo una su cinque: calcolato 0.2, nel libro 0.2


Il gradient boosting si scrive Fₘ = Fₘ₋₁ + η · hₘ. Nel racconto η vale 1 e gli alberi sono tre: ogni vecchio corregge l'errore lasciato dagli altri.

In [11]:
ritardo_vero = 50                    # i minuti di ritardo della Stella
eta = 1
stima = 0
for vecchio, correzione, atteso in [("primo", 30, 30), ("secondo", 15, 45), ("terzo", 5, 50)]:
    stima = stima + eta * correzione
    controlla(f"dopo il {vecchio} vecchio (minuti)", stima, atteso, 0)
    print(f"   errore che resta: {ritardo_vero - stima} minuti")
assert stima == ritardo_vero

✓  dopo il primo vecchio (minuti): calcolato 30, nel libro 30
   errore che resta: 20 minuti
✓  dopo il secondo vecchio (minuti): calcolato 45, nel libro 45
   errore che resta: 5 minuti
✓  dopo il terzo vecchio (minuti): calcolato 50, nel libro 50
   errore che resta: 0 minuti


## Le serie e la forchetta

*Nel racconto: la forchetta «tra trenta e quarantacinque».*

La forchetta si costruisce prevedendo dei quantili: il 10% e il 90% delle uscite possibili.

In [12]:
controlla("giorni su dieci in cui il numero vero cade dentro la forchetta", (0.90 - 0.10) * 10, 8, 1e-9)
assert 30 <= 38 <= 45                # il «trentotto» del racconto sta dentro la forchetta

✓  giorni su dieci in cui il numero vero cade dentro la forchetta: calcolato 8, nel libro 8


## Le linee tra le barche

*Nel racconto: il peschereccio e le stesse tre barche; due o tre giri di messaggi.*

Il peschereccio si affianca a tre barche, e quelle ad altre due. A ogni giro ogni punto ascolta i vicini: dopo un giro sa dei vicini, dopo due dei vicini dei vicini.

In [13]:
linee = [("peschereccio", "barca 1"), ("peschereccio", "barca 2"), ("peschereccio", "barca 3"),
         ("barca 1", "barca nota 1"), ("barca 2", "barca nota 1"), ("barca 3", "barca nota 2")]
vicini = {}
for a, b in linee:
    vicini.setdefault(a, set()).add(b)
    vicini.setdefault(b, set()).add(a)

sa_di = {punto: {punto} for punto in vicini}     # all'inizio ognuno sa solo di sé
for giri, atteso in [("un giro", 3), ("due giri", 5)]:
    sa_di = {punto: sa_di[punto].union(*(sa_di[v] for v in vicini[punto])) for punto in vicini}
    conta(f"dopo {giri}, barche di cui il peschereccio sa", len(sa_di["peschereccio"]) - 1, atteso)

✓  dopo un giro, barche di cui il peschereccio sa: calcolato 3, nel libro 3
✓  dopo due giri, barche di cui il peschereccio sa: calcolato 5, nel libro 5


## Kata per casa

Qui non ci sono conti da rifare: i due esercizi chiedono di scegliere una scuola e di spiegare la diffusione a un amico. Le soluzioni sono in fondo al libro.